In [1]:
import os
import numpy as np
from PIL import Image
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc
from concurrent.futures import ProcessPoolExecutor, as_completed
import psutil
from tqdm import tqdm
from pathlib import Path
import time
import gc

# === Define custom colormap ===
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),
    (0.00000001, "#000000"),
    (0.10, "#000080"),
    (0.15, "#0000FF"),
    (0.30, "#8000FF"),
    (0.45, "#FF0000"),
    (0.60, "#FF8000"),
    (0.75, "#FFFF00"),
    (1.0, "#FFFFFF")
])

# === Parameters ===
base_input_folder = "/home/ajarrah/PhD_Thesis/chapter_4/images/lipids_60_to_20_common"
base_output_folder = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/lipids_common"
skip_color = np.array([0, 191, 0])  # RGB for "#00BF00"
pixel_size_um = 20.0  # each pixel = 20 μm

# === Performance parameters ===
max_workers = None  # None for Auto-detect
reserved_cores = 2
max_memory_percent = 80  # Reduced from 85 for safety
progress_interval = 100
batch_size = 500  # Process images in batches to avoid memory issues

# === Helper functions ===
def parse_filename(filename):
    """Extract gene, sample, vmin, vmax from file name formatted as gene|sample|vmin|vmax.ext"""
    base = os.path.splitext(filename)[0]
    parts = base.split("|")
    if len(parts) != 4:
        raise ValueError(f"Invalid filename format: {filename}")
    gene, sample, vmin, vmax = parts
    return gene, sample, float(vmin), float(vmax)

def build_color_to_value_map(cmap, resolution=10000):
    """Precompute RGB → normalized value mapping for fast lookup"""
    values = np.linspace(0, 1, resolution)
    colors = (cmap(values)[:, :3] * 255).astype(np.uint8)
    return values, colors

def color_to_value_vectorized(rgb_array, cmap_values, cmap_colors):
    """Vectorized color to value conversion for entire image"""
    diff = np.sqrt(np.sum((rgb_array[:, None, :] - cmap_colors[None, :, :]) ** 2, axis=2))
    indices = np.argmin(diff, axis=1)
    return cmap_values[indices]

def get_optimal_workers():
    """Determine optimal number of worker processes"""
    if max_workers is not None:
        return max_workers
    
    cpu_count = os.cpu_count() or 1
    available_cores = max(1, cpu_count - reserved_cores)
    
    mem = psutil.virtual_memory()
    available_mem_gb = (mem.total * max_memory_percent / 100 - mem.used) / (1024**3)
    mem_limited_workers = max(1, int(available_mem_gb / 2))
    
    # Cap at 8 workers for stability
    optimal = min(available_cores, mem_limited_workers, 8)
    return optimal

def process_single_image(args):
    """Process a single image file (runs in parallel)"""
    filename, input_folder, skip_color, cmap_values, cmap_colors, pixel_size_um = args
    
    try:
        gene, sample, vmin, vmax = parse_filename(filename)
        img_path = os.path.join(input_folder, filename)
        
        img = Image.open(img_path).convert("RGB")
        img_arr = np.array(img, dtype=np.uint8)
        height, width, _ = img_arr.shape
        
        img_flat = img_arr.reshape(-1, 3)
        y_coords, x_coords = np.meshgrid(np.arange(height), np.arange(width), indexing='ij')
        y_flat = y_coords.ravel()
        x_flat = x_coords.ravel()
        
        mask = ~np.all(img_flat == skip_color, axis=1)
        
        if not np.any(mask):
            return None
        
        valid_colors = img_flat[mask]
        valid_x = x_flat[mask]
        valid_y = y_flat[mask]
        
        val_norm = color_to_value_vectorized(valid_colors, cmap_values, cmap_colors)
        val = vmin + val_norm * (vmax - vmin)
        
        df = pd.DataFrame({
            "x": valid_x,
            "y": valid_y,
            "intensity": val,
            "gene": gene,
            "sample": sample
        })
        
        return df
        
    except Exception as e:
        return None

def check_memory_usage():
    """Check if memory usage is approaching limit"""
    mem = psutil.virtual_memory()
    if mem.percent > max_memory_percent:
        return True
    return False

def process_batch(image_batch, input_folder, cmap_values, cmap_colors, worker_count, desc="Processing"):
    """Process a batch of images"""
    args_list = [
        (fn, input_folder, skip_color, cmap_values, cmap_colors, pixel_size_um)
        for fn in image_batch
    ]
    
    batch_data = []
    failed_count = 0
    
    try:
        with ProcessPoolExecutor(max_workers=worker_count) as executor:
            futures = {executor.submit(process_single_image, args): args[0] for args in args_list}
            
            with tqdm(total=len(image_batch), desc=desc, unit="img", leave=False) as pbar:
                for future in as_completed(futures):
                    try:
                        result = future.result(timeout=30)  # 30 second timeout per image
                        if result is not None:
                            batch_data.append(result)
                    except Exception as e:
                        failed_count += 1
                    finally:
                        pbar.update(1)
    except Exception as e:
        print(f"⚠️  Batch processing error: {e}")
    
    return batch_data, failed_count

def process_folder(input_folder, output_h5ad, cmap_values, cmap_colors):
    """Process all images in a single folder and save to h5ad"""
    folder_name = os.path.basename(input_folder)
    print(f"\n{'='*80}")
    print(f"📂 Processing folder: {folder_name}")
    print(f"{'='*80}")
    
    start_time = time.time()
    
    # Get list of image files
    image_files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff"))
    ]
    
    if not image_files:
        print(f"⚠️  No image files found in {folder_name}, skipping...")
        return None
    
    print(f"📁 Found {len(image_files)} images to process")
    
    # Determine worker count
    worker_count = get_optimal_workers()
    print(f"🔧 Using {worker_count} workers with batch size of {batch_size}")
    
    # Process images in batches
    all_data = []
    total_failed = 0
    num_batches = (len(image_files) + batch_size - 1) // batch_size
    
    print(f"📦 Processing in {num_batches} batches...")
    
    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, len(image_files))
        image_batch = image_files[start_idx:end_idx]
        
        batch_desc = f"Batch {batch_idx + 1}/{num_batches}"
        print(f"\n{batch_desc}: Processing images {start_idx + 1}-{end_idx} of {len(image_files)}")
        
        batch_data, failed = process_batch(
            image_batch, input_folder, cmap_values, cmap_colors, 
            worker_count, desc=batch_desc
        )
        
        all_data.extend(batch_data)
        total_failed += failed
        
        # Memory cleanup after each batch
        gc.collect()
        
        # Check memory and adjust workers if needed
        mem = psutil.virtual_memory()
        print(f"   Memory: {mem.percent:.1f}% | Processed: {len(batch_data)}/{len(image_batch)} | Failed: {failed}")
        
        if mem.percent > max_memory_percent and worker_count > 1:
            worker_count = max(1, worker_count - 1)
            print(f"   ⚠️  Reducing workers to {worker_count} due to high memory usage")
    
    if not all_data:
        print(f"❌ No valid data extracted from {folder_name}")
        return None
    
    processed_count = len(all_data)
    print(f"\n✅ Successfully processed {processed_count}/{len(image_files)} images ({total_failed} failed)")
    
    # Combine all data
    print("🔄 Combining data from all images...")
    df_all = pd.concat(all_data, ignore_index=True)
    print(f"   Total pixels: {len(df_all):,}")
    
    # Convert pixel coordinates to micrometers
    df_all["x_um"] = df_all["x"] * pixel_size_um
    df_all["y_um"] = df_all["y"] * pixel_size_um
    
    # Pivot to have pixels × genes
    print("📊 Creating gene expression matrix...")
    pivot = df_all.pivot_table(
        index=["sample", "y", "x", "x_um", "y_um"],
        columns="gene",
        values="intensity",
        fill_value=0
    )
    
    # Convert to AnnData
    print("🧬 Building AnnData object...")
    X = pivot.to_numpy()
    obs = pivot.index.to_frame().reset_index(drop=True)
    var = pd.DataFrame(index=pivot.columns)
    
    adata = sc.AnnData(X=X, obs=obs, var=var)
    adata.obsm["spatial"] = adata.obs[["x_um", "y_um"]].to_numpy()
    
    # Save
    print(f"💾 Writing to disk: {output_h5ad}")
    os.makedirs(os.path.dirname(output_h5ad), exist_ok=True)
    adata.write(output_h5ad)
    
    elapsed = time.time() - start_time
    print(f"✅ Saved AnnData with shape {adata.shape}")
    print(f"   📍 Spatial coordinates: adata.obsm['spatial'] (in µm)")
    print(f"   🧬 Genes: {adata.n_vars}, Observations: {adata.n_obs}")
    print(f"   ⏱️  Processing time: {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")
    
    return {
        'folder': folder_name,
        'output': output_h5ad,
        'shape': adata.shape,
        'time': elapsed,
        'processed': processed_count,
        'failed': total_failed
    }

# === Main execution ===
def main():
    print("🚀 Starting batch processing of all subfolders...")
    print(f"📂 Base input folder: {base_input_folder}")
    print(f"💾 Base output folder: {base_output_folder}")
    
    # Create output directory
    os.makedirs(base_output_folder, exist_ok=True)
    
    # Pre-build colormap lookup (shared across all folders)
    print("\n🎨 Building colormap lookup table...")
    cmap_values, cmap_colors = build_color_to_value_map(custom_cmap, resolution=10000)
    
    # Find all subfolders
    subfolders = [
        f for f in os.listdir(base_input_folder)
        if os.path.isdir(os.path.join(base_input_folder, f))
    ]
    
    if not subfolders:
        print(f"❌ No subfolders found in {base_input_folder}")
        return
    
    subfolders.sort()  # Process in alphabetical order
    print(f"\n📁 Found {len(subfolders)} subfolders to process:")
    for sf in subfolders:
        print(f"   • {sf}")
    
    # Process each subfolder
    total_start = time.time()
    results = []
    
    for i, subfolder in enumerate(subfolders, 1):
        print(f"\n\n{'#'*80}")
        print(f"# Processing subfolder {i}/{len(subfolders)}: {subfolder}")
        print(f"{'#'*80}")
        
        input_folder = os.path.join(base_input_folder, subfolder)
        
        # Generate output filename
        output_name = subfolder.lower().replace(' ', '_').replace('-', '_') + "_20.h5ad"
        output_h5ad = os.path.join(base_output_folder, output_name)
        
        # Check if already processed
        if os.path.exists(output_h5ad):
            print(f"⚠️  Output file already exists: {output_h5ad}")
            print("   ⏭️  Skipping (delete file to reprocess)...")
            continue
        
        try:
            result = process_folder(input_folder, output_h5ad, cmap_values, cmap_colors)
            if result:
                results.append(result)
        except Exception as e:
            print(f"❌ Failed to process {subfolder}: {e}")
            continue
        
        # Memory cleanup between folders
        gc.collect()
    
    # Summary
    total_elapsed = time.time() - total_start
    print(f"\n\n{'='*80}")
    print("🎉 BATCH PROCESSING COMPLETE!")
    print(f"{'='*80}")
    print(f"✅ Successfully processed: {len(results)}/{len(subfolders)} folders")
    print(f"⏱️  Total time: {total_elapsed:.1f} seconds ({total_elapsed/60:.1f} minutes)")
    
    if results:
        print(f"\n📊 Summary:")
        for r in results:
            print(f"   • {r['folder']}: {r['shape']} | {r['processed']} images ({r['failed']} failed) | {r['time']:.1f}s")
        print(f"\n💾 All files saved to: {base_output_folder}")
    
    mem = psutil.virtual_memory()
    print(f"\n📊 Final memory usage: {mem.percent:.1f}%")

if __name__ == "__main__":
    main()

🚀 Starting batch processing of all subfolders...
📂 Base input folder: /home/ajarrah/PhD_Thesis/chapter_4/images/lipids_60_to_20_common
💾 Base output folder: /home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/lipids_common

🎨 Building colormap lookup table...

📁 Found 16 subfolders to process:
   • aad_1
   • aad_2
   • aad_3
   • aad_4
   • ac_1
   • ac_2
   • ac_3
   • ac_4
   • yad_1
   • yad_2
   • yad_3
   • yad_4
   • yc_1
   • yc_2
   • yc_3
   • yc_4


################################################################################
# Processing subfolder 1/16: aad_1
################################################################################

📂 Processing folder: aad_1
📁 Found 528 images to process
🔧 Using 8 workers with batch size of 500
📦 Processing in 2 batches...

Batch 1/2: Processing images 1-500 of 528


   Memory: 6.5% | Processed: 500/500 | Failed: 0

Batch 2/2: Processing images 501-528 of 528


   Memory: 6.3% | Processed: 28/28 | Failed: 0

✅ Successfully processed 528/528 images (0 failed)
🔄 Combining data from all images...
   Total pixels: 18,409,248
📊 Creating gene expression matrix...
🧬 Building AnnData object...
💾 Writing to disk: /home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/lipids_common/aad_1_20.h5ad
✅ Saved AnnData with shape (34866, 528)
   📍 Spatial coordinates: adata.obsm['spatial'] (in µm)
   🧬 Genes: 528, Observations: 34866
   ⏱️  Processing time: 729.1 seconds (12.2 minutes)


/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)




################################################################################
# Processing subfolder 2/16: aad_2
################################################################################

📂 Processing folder: aad_2
📁 Found 528 images to process
🔧 Using 8 workers with batch size of 500
📦 Processing in 2 batches...

Batch 1/2: Processing images 1-500 of 528


...



################################################################################
# Processing subfolder 16/16: yc_4
################################################################################

📂 Processing folder: yc_4
📁 Found 528 images to process
🔧 Using 8 workers with batch size of 500
📦 Processing in 2 batches...

Batch 1/2: Processing images 1-500 of 528


   Memory: 7.1% | Processed: 500/500 | Failed: 0

Batch 2/2: Processing images 501-528 of 528


   Memory: 6.1% | Processed: 28/28 | Failed: 0

✅ Successfully processed 528/528 images (0 failed)
🔄 Combining data from all images...
   Total pixels: 17,344,800
📊 Creating gene expression matrix...
🧬 Building AnnData object...
💾 Writing to disk: /home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/lipids_common/yc_4_20.h5ad
✅ Saved AnnData with shape (32850, 528)
   📍 Spatial coordinates: adata.obsm['spatial'] (in µm)
   🧬 Genes: 528, Observations: 32850
   ⏱️  Processing time: 689.9 seconds (11.5 minutes)


/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)




🎉 BATCH PROCESSING COMPLETE!
✅ Successfully processed: 16/16 folders
⏱️  Total time: 12352.9 seconds (205.9 minutes)

📊 Summary:
   • aad_1: (34866, 528) | 528 images (0 failed) | 729.1s
   • aad_2: (32031, 528) | 528 images (0 failed) | 674.0s
   • aad_3: (28233, 528) | 528 images (0 failed) | 590.6s
   • aad_4: (36522, 528) | 528 images (0 failed) | 761.9s
   • ac_1: (37422, 528) | 528 images (0 failed) | 782.8s
   • ac_2: (30447, 528) | 528 images (0 failed) | 637.4s
   • ac_3: (40860, 528) | 528 images (0 failed) | 852.7s
   • ac_4: (41535, 528) | 528 images (0 failed) | 868.5s
   • yad_1: (40437, 528) | 528 images (0 failed) | 843.3s
   • yad_2: (40437, 528) | 528 images (0 failed) | 845.0s
   • yad_3: (36891, 528) | 528 images (0 failed) | 770.2s
   • yad_4: (41184, 528) | 528 images (0 failed) | 862.3s
   • yc_1: (35883, 528) | 528 images (0 failed) | 749.6s
   • yc_2: (42219, 528) | 528 images (0 failed) | 881.8s
   • yc_3: (38682, 528) | 528 images (0 failed) | 808.6s
   • y